### Setup


In [2]:
# Add the parent directory of the current working directory to the Python path at runtime. 
# In order to import modules from the src directory.
import os
import sys 

current_dir = os.getcwd()
parent_dir = os.path.dirname(current_dir)
sys.path.insert(0, parent_dir)

In [3]:
import numpy as np
import pandas as pd
import bambi as bmb
import arviz as az

from prettytable import PrettyTable

from src.stat_utils import *
from src.anl_utils import load_data

### Load and prepare data

In [4]:
sim_results_folder = '../results/simulation'
data_folder = '../data'
sync_at_file = os.path.join(sim_results_folder, 'first_session_arnold_tongues.npy')
emp_at_file = os.path.join(data_folder, 'Experiment.csv')

In [37]:
# Load the simulations results and the empirical data
sync_results = np.load(sync_at_file)
sync_results_vector = sync_results.mean(axis=0).flatten()
data = load_data(emp_at_file)

# Filter data for learning phase (Sessions 1 to 8)
data_learning = data[data['SessionID'] <= 8].copy()

# Map synchrony values to each condition in DataFrame
data_learning['Synchrony'] = data_learning['Condition'].apply(lambda x: sync_results_vector[x-1])

# Z-score the relevant columns
data_learning = zscore_data(data_learning, ['ContrastHeterogeneity', 'GridCoarseness', 'Synchrony'])

# Center the session number
data_learning['SessionID'] = data_learning['SessionID'] - data_learning['SessionID'].mean()

# Dummy code session 9
data['Transfer'] = (data['SessionID'] == 9)


### Define statistical models

In [11]:
model_features = bmb.Model(
    "Correct ~ 1 + ContrastHeterogeneity * GridCoarseness + SessionID * (ContrastHeterogeneity + GridCoarseness) +  (1 + SessionID + ContrastHeterogeneity * GridCoarseness|SubjectID)",
    data=data,
    family="bernoulli"
)

Does learning occur (i.e., does session have an effect on performance)? 

Do the effects of contrast heterogeneity and/or grid coarseness depend on session?

In [12]:
idata_features = model_features.fit(
    draws=2000, tune=2000, target_accept=0.9,
    idata_kwargs={"log_likelihood": True}, progressbar=False
)

Modeling the probability that Correct==1
Initializing NUTS using jitter+adapt_diag...
Multiprocess sampling (4 chains in 4 jobs)
NUTS: [Intercept, ContrastHeterogeneity, GridCoarseness, ContrastHeterogeneity:GridCoarseness, SessionID, SessionID:ContrastHeterogeneity, SessionID:GridCoarseness, 1|SubjectID_sigma, 1|SubjectID_offset, SessionID|SubjectID_sigma, SessionID|SubjectID_offset, ContrastHeterogeneity|SubjectID_sigma, ContrastHeterogeneity|SubjectID_offset, GridCoarseness|SubjectID_sigma, GridCoarseness|SubjectID_offset, ContrastHeterogeneity:GridCoarseness|SubjectID_sigma, ContrastHeterogeneity:GridCoarseness|SubjectID_offset]
Sampling 4 chains for 2_000 tune and 2_000 draw iterations (8_000 + 8_000 draws total) took 12006 seconds.
There were 52 divergences after tuning. Increase `target_accept` or reparameterize.


In [15]:
predictors = ["SessionID", "ContrastHeterogeneity", "GridCoarseness", "SessionID:ContrastHeterogeneity","SessionID:GridCoarseness", "ContrastHeterogeneity:GridCoarseness"]
directions = ['greater', 'less', 'less','less','greater', 'greater']

posterior = posterior_table(idata_features, predictors, directions)

odds_ratios = OR_table(idata_features, predictors)

print("One-sided posterior probabilities:")
print(posterior)
print("\nOdds ratios:")
print(odds_ratios)

az.summary(idata_features, var_names=predictors, hdi_prob=0.95)

One-sided posterior probabilities:
+--------------------------------------+-----------+-------+
|              Predictor               | direction |   P   |
+--------------------------------------+-----------+-------+
|              SessionID               |  greater  | 1.000 |
|        ContrastHeterogeneity         |    less   | 1.000 |
|            GridCoarseness            |    less   | 1.000 |
|   SessionID:ContrastHeterogeneity    |    less   | 1.000 |
|       SessionID:GridCoarseness       |  greater  | 0.410 |
| ContrastHeterogeneity:GridCoarseness |  greater  | 1.000 |
+--------------------------------------+-----------+-------+

Odds ratios:
+--------------------------------------+--------+--------------+---------------+
|              Predictor               |  Mean  | Lower (2.5%) | Upper (97.5%) |
+--------------------------------------+--------+--------------+---------------+
|              SessionID               | 1.135  |    1.063     |     1.206     |
|        Contrast

,mean,sd,hdi_2.5%,hdi_97.5%,mcse_mean,mcse_sd,ess_bulk,ess_tail,r_hat
SessionID,0.126,0.032,0.064,0.189,0.000,0.000,5259.0,3590.0,1.0
ContrastHeterogeneity,-7.158,0.482,-8.141,-6.253,0.009,0.009,3259.0,3649.0,1.0
GridCoarseness,-3.785,0.237,-4.244,-3.311,0.004,0.004,3674.0,2964.0,1.0
SessionID:ContrastHeterogeneity,-0.136,0.013,-0.160,-0.111,0.000,0.000,8815.0,6091.0,1.0
SessionID:GridCoarseness,-0.005,0.023,-0.050,0.040,0.000,0.000,5755.0,4385.0,1.0
ContrastHeterogeneity:GridCoarseness,4.200,0.256,3.692,4.687,0.004,0.006,4314.0,2946.0,1.0


In [ ]:
import numpy as np
import pandas as pd
import arviz as az

def session_simple_effects_CH(idata, sessions, hdi=0.95):
    # Extract posterior draws for CH main and CH:Session interaction
    posterior = az.extract(idata, var_names=["ContrastHeterogeneity",
                                        "SessionID:ContrastHeterogeneity"]).to_dataframe()
    beta_contrast_heterogeneity   = posterior["ContrastHeterogeneity"].to_numpy()
    beta_session_ch_interaction = posterior["SessionID:ContrastHeterogeneity"].to_numpy()

    rows = []
    for session in sessions:
        beta_draws = beta_contrast_heterogeneity + beta_session_ch_interaction  * session
        odds_ratio_draws   = np.exp(beta_draws)

        hdi_low, hdi_high = az.hdi(beta_draws, hdi_prob=hdi)
        odds_ratio_low, odds_ratio_high = az.hdi(odds_ratio_draws, hdi_prob=hdi)

        rows.append({
            "Session": session,
            "beta_mean": float(beta_draws.mean()),
            f"beta_hdi_{int((1-hdi)/2*100)}%": float(hdi_low),
            f"beta_hdi_{int((1+hdi)/2*100)}%": float(hdi_high),
            "OR_mean": float(odds_ratio_draws.mean()),
            f"OR_hdi_{int((1-hdi)/2*100)}%": float(odds_ratio_low),
            f"OR_hdi_{int((1+hdi)/2*100)}%": float(odds_ratio_high),
            "Pr_beta_less_0": float((beta_draws < 0).mean())
        })
    return pd.DataFrame(rows)

# Example: sessions 1..8
sessions = list(range(1, 9))
tbl_ch_by_session = session_simple_effects_CH(idata_features, sessions, hdi=0.95)
print(tbl_ch_by_session)


   Session  beta_mean  beta_hdi_2%  beta_hdi_97%   OR_mean  OR_hdi_2%  \
0        1  -7.293833    -8.263997     -6.386387  0.000770   0.000196   
1        2  -7.430046    -8.410234     -6.535753  0.000672   0.000166   
2        3  -7.566259    -8.522559     -6.656934  0.000586   0.000143   
3        4  -7.702472    -8.656554     -6.791785  0.000511   0.000122   
4        5  -7.838685    -8.793335     -6.926476  0.000446   0.000105   
5        6  -7.974898    -8.925605     -7.055919  0.000389   0.000099   
6        7  -8.111111    -9.031632     -7.157665  0.000340   0.000077   
7        8  -8.247324    -9.217017     -7.337810  0.000297   0.000070   

   OR_hdi_97%  Pr_beta_less_0  
0    0.001532             1.0  
1    0.001331             1.0  
2    0.001160             1.0  
3    0.001013             1.0  
4    0.000881             1.0  
5    0.000777             1.0  
6    0.000669             1.0  
7    0.000590             1.0  


In [36]:
import numpy as np
import arviz as az

# 1) Build the session rows (do NOT change SubjectID)
def get_session_rows(df, session):
    return df[df["SessionID"] == session].copy()

# 2) Posterior mean accuracy per draw, using posterior predictive samples
def post_mean_prob_via_pps(model, idata, new_data):
    # Ask Bambi for posterior predictive samples for the rows in new_data
    # Returns an InferenceData with group: posterior_predictive
    pp = model.predict(idata=idata, data=new_data, kind="pps", inplace=False)

    # Find the outcome var name in posterior_predictive (usually your response name, e.g., "Correct")
    varname = list(pp.posterior_predictive.data_vars)[0]

    # Extract to numpy: shape (chain, draw, obs)
    arr = pp.posterior_predictive[varname].values

    # Flatten chains x draws -> draws
    draws = arr.shape[0] * arr.shape[1]
    arr2 = arr.reshape(draws, arr.shape[2])  # (draws, obs)

    # Average across obs -> one mean accuracy per draw
    p = arr2.mean(axis=1)  # shape: (draws,)
    return p

# --- compute session-level posterior mean accuracy for S9 and S2 ---

rows_s9 = get_session_rows(data, session=4)  # transfer, untrained
rows_s2 = get_session_rows(data, session=-3)  # early, trained

p_s9 = post_mean_prob_via_pps(model_features, idata_features, rows_s9)
p_s2 = post_mean_prob_via_pps(model_features, idata_features, rows_s2)

# Posterior probability that transfer < early
prob_s9_lt_s2 = float((p_s9 < p_s2).mean())

# Posterior of the difference on the probability scale
delta = p_s9 - p_s2
hdi = az.hdi(delta, hdi_prob=0.95)

print(f"P(Session 9 < Session 2) = {prob_s9_lt_s2:.3f}")
print(f"Δ (S9 − S2) mean = {delta.mean():.3f}, 95% CrI [{hdi[0]:.3f}, {hdi[1]:.3f}]")


P(Session 9 < Session 2) = 0.009
Δ (S9 − S2) mean = 0.016, 95% CrI [0.002, 0.030]


In [58]:
import numpy as np
import pandas as pd
import arviz as az

# 0) Identify the data used to fit the model (sessions 1–8, already available)
# If you didn't keep a separate variable, reconstruct from your current `data`:
train_mask = (data["SessionID"] >= 1) & (data["SessionID"] <= 8)

# 1) Save training-set scaling parameters
CH_mean = data.loc[train_mask, "ContrastHeterogeneity"].mean()
CH_sd   = data.loc[train_mask, "ContrastHeterogeneity"].std(ddof=0)

GC_mean = data.loc[train_mask, "GridCoarseness"].mean()
GC_sd   = data.loc[train_mask, "GridCoarseness"].std(ddof=0)

sess_center = data.loc[train_mask, "SessionID"].mean()  # e.g., 4.5 if 1..8

def apply_fit_transforms(df_new: pd.DataFrame) -> pd.DataFrame:
    df_new = df_new.copy()
    # z-score CH & GC with training params
    df_new["ContrastHeterogeneity"] = (df_new["ContrastHeterogeneity"] - CH_mean) / CH_sd
    df_new["GridCoarseness"]        = (df_new["GridCoarseness"]        - GC_mean) / GC_sd
    # center Session with training center
    df_new["SessionID"] = df_new["SessionID"] - sess_center
    return df_new

# Your helper to collect session rows (keep real SubjectID values)
def get_session_rows(df, session):
    return df[df["SessionID"] == session].copy()

# Use the same PPS-based function from before
def post_mean_prob_via_pps(model, idata, new_data):
    # IMPORTANT: new_data must already be transformed to match the fit
    pp = model.predict(idata=idata, data=new_data, kind="pps", inplace=False)
    varname = list(pp.posterior_predictive.data_vars)[0]  # typically "Correct"
    arr = pp.posterior_predictive[varname].values  # shape: (chain, draw, obs)
    draws = arr.shape[0] * arr.shape[1]
    arr2 = arr.reshape(draws, arr.shape[2])        # (draws, obs)
    return arr2.mean(axis=1)                       # one mean accuracy per draw

# --- Build, transform, and predict for S9 vs S2 ---

rows_s9_raw = get_session_rows(data, session=9)   # untransformed slice
rows_s2_raw = get_session_rows(data, session=8)

rows_s9 = apply_fit_transforms(rows_s9_raw)
rows_s2 = apply_fit_transforms(rows_s2_raw)

p_s9 = post_mean_prob_via_pps(model_features, idata_features, rows_s9)
p_s2 = post_mean_prob_via_pps(model_features, idata_features, rows_s2)

prob_s9_lt_s2 = float((p_s9 < p_s2).mean())
delta = p_s9 - p_s2
hdi = az.hdi(delta, hdi_prob=0.95)

print(f"P(Session 9 < Session 8) = {prob_s9_lt_s2:.3f}")
print(f"Δ (S9 − S8) mean = {delta.mean():.3f}, 95% CrI [{hdi[0]:.3f}, {hdi[1]:.3f}]")


P(Session 9 < Session 8) = 0.340
Δ (S9 − S8) mean = 0.002, 95% CrI [-0.007, 0.010]


In [54]:
# get mean of Correct for SessionID=9
data[data['SessionID'] == 2]['Correct'].mean()

np.float64(0.6995)

In [57]:
data[data['SessionID'] == 8]['Correct'].mean()

np.float64(0.7376666666666667)

In [56]:
data[data['SessionID'] == 9]['Correct'].mean()

np.float64(0.6881666666666667)

In [31]:
import numpy as np
import pandas as pd
import pymc as pm

from scipy.special import expit
import arviz as az

# --- helpers ---
def make_pop_rows(df, session):
    sub = df[df["SessionID"] == session].copy()
    # use the empirical CH×GC mix of that session
    # make a dummy SubjectID so predictions are at population level (RE = 0)
    sub["SubjectID"] = sub["SubjectID"].iloc[0]
    # keep only the columns needed by the model formula
    keep = ["SubjectID", "SessionID", "ContrastHeterogeneity", "GridCoarseness"]
    return sub[keep]

def post_mean_prob(model, idata, new_data):
    # get PyMC model and posterior
    with model.backend.model:
        ppc = pm.sample_posterior_predictive(
            idata,
            var_names=["likelihood"],  # "likelihood" is the outcome node in Bambi
            predictions=True,
            extend_inferencedata=False,
            random_seed=42,
            keep_size=True,
            data=new_data
        )

    # ppc["likelihood"] has shape (chains, draws, rows)
    arr = ppc["likelihood"]
    # average over rows -> one prob per draw
    p = arr.mean(axis=-1).reshape(-1)
    return p

# build rows for Session 9 (transfer) and Session 2 (trained)
rows_s9 = make_pop_rows(data, session=9)
rows_s2 = make_pop_rows(data, session=2)

p_s9 = post_mean_prob(model_features, idata_features, rows_s9)
p_s2 = post_mean_prob(model_features, idata_features, rows_s2)

prob_s9_lt_s2 = (p_s9 < p_s2).mean()
delta = p_s9 - p_s2

print("Pr(S9 < S2) =", prob_s9_lt_s2)
print("Delta mean =", np.mean(delta))
print("95% HDI =", az.hdi(delta, hdi_prob=0.95).to_dict())


TypeError: sample_posterior_predictive() got an unexpected keyword argument 'keep_size'